# K-means and Hierarchical Clustering Analysis
## Real Estate Project Data Visualization

This notebook performs K-means and hierarchical clustering on real estate data with comprehensive visualizations including elbow curves, dendrograms, scatter plots, and cluster comparisons.

In [ ]:
# 1. Import Visualization and Clustering Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

In [ ]:
# 2. Load and Prepare Dataset
df = pd.read_csv('Real_Estate_Project.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names and types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# 3. Data Cleaning and Feature Selection
# Remove $ signs and convert to numeric
df['sale_price_numeric'] = df['sale_price'].str.replace('$', '').str.replace(',', '').astype(float)

# Select numeric features for clustering
clustering_features = ['floor_area_sqft', 'sale_price_numeric', 'satisfaction']

# Create a clean dataset with no missing values
df_clean = df[clustering_features].dropna()

print(f"Clean dataset shape: {df_clean.shape}")
print("\nFeature statistics:")
print(df_clean.describe())

# Standardize the features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(df_clean)
features_df = pd.DataFrame(features_scaled, columns=clustering_features)

print("\nScaled features (first 5 rows):")
print(features_df.head())

In [ ]:
# 4. K-means Clustering - Elbow Method
# Test different numbers of clusters
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_scaled, kmeans.labels_))
    davies_bouldin_scores.append(davies_bouldin_score(features_scaled, kmeans.labels_))

# Create Elbow Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Inertia plot
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method - Inertia', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette Score plot
axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score (Higher is Better)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Davies-Bouldin Score plot
axes[2].plot(K_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
axes[2].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[2].set_ylabel('Davies-Bouldin Score', fontsize=12)
axes[2].set_title('Davies-Bouldin Score (Lower is Better)', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Clustering Metrics:")
for i, k in enumerate(K_range):
    print(f"k={k}: Inertia={inertias[i]:.2f}, Silhouette={silhouette_scores[i]:.4f}, Davies-Bouldin={davies_bouldin_scores[i]:.4f}")

In [ ]:
# 5. Apply K-means with Optimal k (k=3)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(features_scaled)

# Add cluster labels to original dataframe
df_clean['KMeans_Cluster'] = kmeans_labels

# Create 2D visualization using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
features_pca = pca.fit_transform(features_scaled)

# K-means Scatter Plot
plt.figure(figsize=(10, 7))
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1], c=kmeans_labels, 
                     cmap='viridis', s=100, alpha=0.6, edgecolors='black')
plt.scatter(pca.transform(kmeans.cluster_centers_)[:, 0], 
           pca.transform(kmeans.cluster_centers_)[:, 1],
           c='red', marker='X', s=300, edgecolors='black', linewidth=2, label='Centroids')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
plt.title('K-means Clustering (k=3)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nK-means Clustering Results (k={optimal_k}):")
print(f"Silhouette Score: {silhouette_score(features_scaled, kmeans_labels):.4f}")

In [ ]:
# 6. Hierarchical Clustering - Dendrogram
# Use a sample of data for dendrogram (too many samples makes it unreadable)
sample_size = 50
sample_indices = np.random.choice(len(features_scaled), sample_size, replace=False)
features_sample = features_scaled[sample_indices]

# Calculate linkage matrix using different methods
plt.figure(figsize=(14, 6))

linkage_matrix = linkage(features_sample, method='ward')

# Create dendrogram
dendrogram(linkage_matrix, 
          truncate_mode=None,
          color_threshold=0,
          above_threshold_color='gray')
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Create dendrograms with different linkage methods
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
methods = ['ward', 'complete', 'average', 'single']

for idx, method in enumerate(methods):
    ax = axes[idx // 2, idx % 2]
    linkage_mat = linkage(features_sample, method=method)
    dendrogram(linkage_mat, ax=ax, color_threshold=0, above_threshold_color='gray')
    ax.set_title(f'Hierarchical Clustering ({method.capitalize()} Linkage)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Distance')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# 7. Apply Hierarchical Clustering with k=3 (Ward linkage)
hierarchical = AgglomerativeClustering(n_clusters=3, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(features_scaled)

# Add cluster labels
df_clean['Hierarchical_Cluster'] = hierarchical_labels

# Visualization
plt.figure(figsize=(10, 7))
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1], c=hierarchical_labels, 
                     cmap='plasma', s=100, alpha=0.6, edgecolors='black')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
plt.title('Hierarchical Clustering (Ward Linkage, k=3)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Hierarchical Clustering Results (k=3, Ward Linkage):")
print(f"Silhouette Score: {silhouette_score(features_scaled, hierarchical_labels):.4f}")

In [ ]:
# 8. Cluster Comparison - Side by Side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# K-means
scatter1 = axes[0].scatter(features_pca[:, 0], features_pca[:, 1], c=kmeans_labels, 
                          cmap='viridis', s=100, alpha=0.6, edgecolors='black')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
axes[0].set_title('K-means Clustering', fontsize=13, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')
axes[0].grid(True, alpha=0.3)

# Hierarchical
scatter2 = axes[1].scatter(features_pca[:, 0], features_pca[:, 1], c=hierarchical_labels, 
                          cmap='plasma', s=100, alpha=0.6, edgecolors='black')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
axes[1].set_title('Hierarchical Clustering', fontsize=13, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 9. Cluster Profiles - K-means
print("=" * 70)
print("K-MEANS CLUSTER PROFILES")
print("=" * 70)

kmeans_profiles = df_clean.groupby('KMeans_Cluster').agg({
    'floor_area_sqft': ['mean', 'min', 'max', 'std'],
    'sale_price_numeric': ['mean', 'min', 'max', 'std'],
    'satisfaction': ['mean', 'min', 'max', 'std']
}).round(2)

print(kmeans_profiles)
print(f"\nCluster Sizes:\n{df_clean['KMeans_Cluster'].value_counts().sort_index()}")

# Visualization of cluster profiles
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(['floor_area_sqft', 'sale_price_numeric', 'satisfaction']):
    cluster_means = df_clean.groupby('KMeans_Cluster')[feature].mean()
    axes[i].bar(cluster_means.index, cluster_means.values, color=['#440154', '#31688e', '#35b779'])
    axes[i].set_xlabel('Cluster', fontsize=12)
    axes[i].set_ylabel('Mean Value', fontsize=12)
    axes[i].set_title(f'K-means: Mean {feature.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    axes[i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# 10. Cluster Profiles - Hierarchical
print("=" * 70)
print("HIERARCHICAL CLUSTER PROFILES")
print("=" * 70)

hier_profiles = df_clean.groupby('Hierarchical_Cluster').agg({
    'floor_area_sqft': ['mean', 'min', 'max', 'std'],
    'sale_price_numeric': ['mean', 'min', 'max', 'std'],
    'satisfaction': ['mean', 'min', 'max', 'std']
}).round(2)

print(hier_profiles)
print(f"\nCluster Sizes:\n{df_clean['Hierarchical_Cluster'].value_counts().sort_index()}")

# Visualization of cluster profiles
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(['floor_area_sqft', 'sale_price_numeric', 'satisfaction']):
    cluster_means = df_clean.groupby('Hierarchical_Cluster')[feature].mean()
    axes[i].bar(cluster_means.index, cluster_means.values, color=['#440154', '#31688e', '#35b779'])
    axes[i].set_xlabel('Cluster', fontsize=12)
    axes[i].set_ylabel('Mean Value', fontsize=12)
    axes[i].set_title(f'Hierarchical: Mean {feature.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    axes[i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# 11. Box Plots - Feature Distribution by Cluster
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features_list = ['floor_area_sqft', 'sale_price_numeric', 'satisfaction']

# K-means box plots
for i, feature in enumerate(features_list):
    data_to_plot = [df_clean[df_clean['KMeans_Cluster'] == j][feature].values for j in range(3)]
    axes[0, i].boxplot(data_to_plot, labels=[f'C{j}' for j in range(3)])
    axes[0, i].set_ylabel(feature.replace('_', ' ').title(), fontsize=11)
    axes[0, i].set_title(f'K-means: {feature.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    axes[0, i].grid(True, alpha=0.3, axis='y')

# Hierarchical box plots
for i, feature in enumerate(features_list):
    data_to_plot = [df_clean[df_clean['Hierarchical_Cluster'] == j][feature].values for j in range(3)]
    axes[1, i].boxplot(data_to_plot, labels=[f'C{j}' for j in range(3)])
    axes[1, i].set_ylabel(feature.replace('_', ' ').title(), fontsize=11)
    axes[1, i].set_title(f'Hierarchical: {feature.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    axes[1, i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# 12. Interactive 3D Scatter Plot with Plotly
# Create 3D scatter for K-means
fig_kmeans = go.Figure(data=[go.Scatter3d(
    x=df_clean['floor_area_sqft'],
    y=df_clean['sale_price_numeric'],
    z=df_clean['satisfaction'],
    mode='markers',
    marker=dict(
        size=5,
        color=kmeans_labels,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cluster")
    ),
    text=[f"Cluster: {c}" for c in kmeans_labels],
    hoverinfo='x+y+z+text'
)])

fig_kmeans.update_layout(
    title='K-means Clustering - 3D Interactive View',
    scene=dict(
        xaxis_title='Floor Area (sqft)',
        yaxis_title='Sale Price ($)',
        zaxis_title='Satisfaction'
    ),
    width=900,
    height=700
)
fig_kmeans.show()

# Create 3D scatter for Hierarchical
fig_hier = go.Figure(data=[go.Scatter3d(
    x=df_clean['floor_area_sqft'],
    y=df_clean['sale_price_numeric'],
    z=df_clean['satisfaction'],
    mode='markers',
    marker=dict(
        size=5,
        color=hierarchical_labels,
        colorscale='Plasma',
        showscale=True,
        colorbar=dict(title="Cluster")
    ),
    text=[f"Cluster: {c}" for c in hierarchical_labels],
    hoverinfo='x+y+z+text'
)])

fig_hier.update_layout(
    title='Hierarchical Clustering - 3D Interactive View',
    scene=dict(
        xaxis_title='Floor Area (sqft)',
        yaxis_title='Sale Price ($)',
        zaxis_title='Satisfaction'
    ),
    width=900,
    height=700
)
fig_hier.show()

In [ ]:
# 13. Heatmap - Cluster Comparison Matrix
from sklearn.preprocessing import MinMaxScaler

# Calculate mean values for each cluster and feature
kmeans_cluster_stats = df_clean.groupby('KMeans_Cluster')[['floor_area_sqft', 'sale_price_numeric', 'satisfaction']].mean()
hier_cluster_stats = df_clean.groupby('Hierarchical_Cluster')[['floor_area_sqft', 'sale_price_numeric', 'satisfaction']].mean()

# Normalize for heatmap
scaler_heat = MinMaxScaler()
kmeans_normalized = pd.DataFrame(
    scaler_heat.fit_transform(kmeans_cluster_stats),
    index=kmeans_cluster_stats.index,
    columns=kmeans_cluster_stats.columns
)
hier_normalized = pd.DataFrame(
    scaler_heat.fit_transform(hier_cluster_stats),
    index=hier_cluster_stats.index,
    columns=hier_cluster_stats.columns
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# K-means heatmap
sns.heatmap(kmeans_normalized.T, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[0], 
            cbar_kws={'label': 'Normalized Value'})
axes[0].set_title('K-means Cluster Profiles (Normalized)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cluster', fontsize=11)
axes[0].set_ylabel('Feature', fontsize=11)

# Hierarchical heatmap
sns.heatmap(hier_normalized.T, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1], 
            cbar_kws={'label': 'Normalized Value'})
axes[1].set_title('Hierarchical Cluster Profiles (Normalized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cluster', fontsize=11)
axes[1].set_ylabel('Feature', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# 14. Summary and Model Comparison
print("=" * 70)
print("CLUSTERING ALGORITHM COMPARISON")
print("=" * 70)

# Calculate comparison metrics
kmeans_silhouette = silhouette_score(features_scaled, kmeans_labels)
kmeans_davies = davies_bouldin_score(features_scaled, kmeans_labels)

hier_silhouette = silhouette_score(features_scaled, hierarchical_labels)
hier_davies = davies_bouldin_score(features_scaled, hierarchical_labels)

comparison_df = pd.DataFrame({
    'Metric': ['Silhouette Score', 'Davies-Bouldin Score'],
    'K-means': [kmeans_silhouette, kmeans_davies],
    'Hierarchical': [hier_silhouette, hier_davies]
}).set_index('Metric')

print("\nClustering Performance Metrics:")
print(comparison_df.to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Silhouette scores
methods = ['K-means', 'Hierarchical']
silhouette_vals = [kmeans_silhouette, hier_silhouette]
colors = ['#440154', '#31688e']
axes[0].bar(methods, silhouette_vals, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Silhouette Score', fontsize=12)
axes[0].set_title('Silhouette Score Comparison\n(Higher is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(silhouette_vals):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

# Davies-Bouldin scores
davies_vals = [kmeans_davies, hier_davies]
axes[1].bar(methods, davies_vals, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Davies-Bouldin Score', fontsize=12)
axes[1].set_title('Davies-Bouldin Score Comparison\n(Lower is Better)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(davies_vals):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print(f"✓ Both methods identified 3 clusters in the real estate data")
print(f"✓ K-means Silhouette Score: {kmeans_silhouette:.4f}")
print(f"✓ Hierarchical Silhouette Score: {hier_silhouette:.4f}")
print(f"✓ Lower Davies-Bouldin is better: K-means={kmeans_davies:.4f}, Hierarchical={hier_davies:.4f}")